## Build Results Fact
1. Read silver results table
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns
    - is_win -> indicates that the driver own the race
    - is_podium -> indicates that the driver own the race
    - has_points -> indicates that the driver has scored points
6. Write the transformed data to gold fact_session_results table

In [0]:
%run ../00-common/01.environment-config


In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
  - `silver.results`
  - `silver.sprints`

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .withColumn("session_type", F.lit("RACE"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
         .withColumn("session_type", F.lit("SPRINT"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

#### Step 2 - UNION `results` and `sprints`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
results_sprints_df.select("finish_position").distinct().show()

#### Step 3 - Add dervied columns
1. is_win -> Indicates that the driver own the race
1. is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
1. has_points -> Indicates that the driver has scored points


In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

#### Step 4 - Write the transformed data to the `gold` `fact_session_results` table

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)